In [3]:
# save as show_tree.py
import os

def print_tree(start_path='.', max_depth=2, prefix=''):
    def _print(current_path, depth, prefix):
        if depth > max_depth:
            return
        files = sorted(os.listdir(current_path))
        for i, file in enumerate(files):
            path = os.path.join(current_path, file)
            connector = "└── " if i == len(files) - 1 else "├── "
            print(prefix + connector + file)
            if os.path.isdir(path):
                extension = "    " if i == len(files) - 1 else "│   "
                _print(path, depth + 1, prefix + extension)

    print(start_path)
    _print(start_path, 1, '')

if __name__ == "__main__":
    print_tree('../FYS3033DL', max_depth=2)


../FYS3033DL
├── .git
│   ├── COMMIT_EDITMSG
│   ├── FETCH_HEAD
│   ├── HEAD
│   ├── ORIG_HEAD
│   ├── config
│   ├── description
│   ├── hooks
│   ├── index
│   ├── info
│   ├── logs
│   ├── objects
│   └── refs
├── .gitignore
├── .ipynb_checkpoints
│   ├── home_exam-checkpoint.ipynb
│   ├── homie_v1-checkpoint.ipynb
│   ├── homie_v2-checkpoint.ipynb
│   ├── report-checkpoint.pdf
│   └── test0404-checkpoint.ipynb
├── FYS3033_homexam.pdf
├── __pycache__
│   └── utils.cpython-310.pyc
├── data
│   ├── .DS_Store
│   ├── __MACOSX
│   ├── problem2
│   └── problem3
├── doc
│   ├── report.aux
│   ├── report.fdb_latexmk
│   ├── report.fls
│   ├── report.log
│   ├── report.out
│   ├── report.pdf
│   ├── report.synctex.gz
│   └── report.tex
├── home_exam.ipynb
├── homie_v1.ipynb
├── logs
├── models
├── plots
├── src
│   ├── .ipynb_checkpoints
│   ├── __pycache__
│   ├── evaluation.py
│   ├── old
│   ├── train.py
│   ├── utils.py
│   └── vgg11bn.py
├── structurefys3033dl.txt
└── test0404.ipynb


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Add the src folder to the Python path
sys.path.append(os.path.join(os.getcwd(), 'src/'))

training_data = np.load ('data/problem2/training_data.npz')
training_images = training_data ['a' ]
training_labels = training_data ['b']

print('training_images shape:', training_images.shape)
print('training_labels shape:', training_labels.shape)
# converting to dataframe
df = pd.DataFrame({
    'images': list(training_images), # Convert numpy array to list for DataFrame
    'labels': training_labels # Keep labels as is
})

# Display the first few rows of the DataFrame
print(df.head())

import matplotlib.gridspec as gridspec
from collections import Counter

counter = Counter(training_labels)

fig = plt.figure(figsize=(20, 10))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1.2])

# Top row: display 5 sample images in a row
gs_top = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=gs[0])
for i in range(5):
    ax = fig.add_subplot(gs_top[i])
    img = training_images[i].transpose(1, 2, 0)  # (C, H, W) -> (H, W, C)
    ax.imshow(img)
    ax.set_title(f"Label: {training_labels[i]}")
    ax.axis('off')

# Bottom row: bar chart for the class distribution
ax_bar = fig.add_subplot(gs[1])
ax_bar.bar(list(counter.keys()), list(counter.values()))
ax_bar.set_title("Class Distribution in Training Set")
ax_bar.set_xticks([0, 1, 2])
ax_bar.set_xticklabels(['Plane', 'Ship', 'Truck'])

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split



# Split indices
train_idx, val_idx = train_test_split(
    np.arange(len(training_labels)),
    test_size=0.2,
    stratify=training_labels,
    random_state=42
)

# Create sets
train_images = training_images[train_idx]
train_labels = training_labels[train_idx]
val_images = training_images[val_idx]
val_labels = training_labels[val_idx]


In [ ]:
from torch.utils.data import DataLoader
from src.utils import ImageDataset

# Create datasets
train_dataset = ImageDataset(train_images, train_labels)
val_dataset = ImageDataset(val_images, val_labels)

# Create loaders
dataloaders = {
    'train': DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,           #  Shuffles training data each epoch
        pin_memory=True         #  Speeds up transfer to GPU (if using CUDA)
    ),
    'val': DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,          #  Keeps validation consistent across epochs
        pin_memory=True         #  Still helpful for val on GPU
    )
}



In [ ]:
from src.vgg11bn import VGG11BN
from src.train import train_model
from src.utils import ImageDataset, set_seed
from src.evaluation import classification_summary, plot_confusion_matrix
import torch.optim as optim
import torch.nn as nn
import torch
from torch.optim.lr_scheduler import StepLR

set_seed(42) # Set seed for reproducibility!

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# cudnn.benchmark = True

# Create model (no dropout for 2a)
model2a = VGG11BN(num_classes=3, dropout=False).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2a.parameters(), lr=1e-4)

# Learning rate scheduler
# StepLR reduces the learning rate by a factor of gamma every step_size epochs
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

# Train
trained_model = train_model(
    model_name='VGG11BN_2a_no_dropout',
    model2a,
    dataloaders,
    criterion,
    optimizer,
    scheduler=None,
    device=device,
    num_epochs=100
)

print("\nModel without Dropout training complete.")

torch.save(trained_model.state_dict(), 'model_without_dropout.pth')

# evaluation
val_loss, val_acc = classification_summary(trained_model, dataloaders['val'], criterion, device)
print(f"Final validation accuracy: {val_acc:.4f}")
print(f"Final validation loss: {val_loss:.4f}")

# Plot confusion matrix
plot_confusion_matrix(trained_model, dataloaders['val'], device, class_names=['Plane', 'Ship', 'Truck'])


In [ ]:
# 2c 

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# cudnn.benchmark = True

# Create model (no dropout for 2a)
model2c = VGG11BN(num_classes=3, dropout=False).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2c.parameters(), lr=1e-4)

# Learning rate scheduler
# StepLR reduces the learning rate by a factor of gamma every step_size epochs
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

# Train
trained_model = train_model(
    model_name='VGG11BN_2c_with_dropout',
    model2c,
    dataloaders,
    criterion,
    optimizer,
    scheduler=None,
    device=device,
    num_epochs=100
)

print("\nModel without Dropout training complete.")

torch.save(trained_model.state_dict(), 'model_without_dropout.pth')

# evaluation
val_loss, val_acc = classification_summary(trained_model, dataloaders['val'], criterion, device)
print(f"Final validation accuracy: {val_acc:.4f}")
print(f"Final validation loss: {val_loss:.4f}")

# Plot confusion matrix
plot_confusion_matrix(trained_model, dataloaders['val'], device, class_names=['Plane', 'Ship', 'Truck'])
